# Table 2 (Instruction-Consistent Build)

This notebook rebuilds Table 2 using the three-step construction in `src/data instruction.txt`:
1. Build issuer totals (from the existing `table1.ipynb` construction pipeline).
2. Build foreign holdings by investor from IMF PIP bilateral data.
3. Construct domestic holdings as issuer totals minus foreign holdings, then assign domestic holdings to the home investor.

Important implementation notes:
- No hard-coded target values are used.
- Confidential/small positive holdings (`0 < value_usd < 500000`) are set to missing before aggregation.
- A separate pseudo-investor row `Reserves` is added from reserve-sector bilateral positions.
- Consistency checks are printed so discrepancies are explicit and auditable.

In [ ]:
from pathlib import Path
from typing import Optional
import json
import pandas as pd

In [8]:
TARGET_YEAR = 2020
SMALL_HOLDING_USD = 500_000.0

project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent

data_dir = project_root / "data"
src_dir = project_root / "src"
table1_nb_path = src_dir / "table1.ipynb"
bilat_path = data_dir / "pip_bilateral_positions.parquet"
local_alloc_path = data_dir / "pip_local_foreign_allocated.parquet"
reserve_path = data_dir / "pip_bilateral_positions_reserve.parquet"
reserve_agg_path = data_dir / "pip_bilateral_positions_reserve_aggregate.parquet"

for p in [table1_nb_path, bilat_path, local_alloc_path, reserve_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

ASSET_CODE_TO_LABEL = {
    "ST_DEBT": "Short-term debt",
    "LT_DEBT": "Long-term debt",
    "EQUITY": "Equity",
}

ASSET_LABEL_TO_MN_COL = {
    "ST_DEBT": "st_mn",
    "LT_DEBT": "lt_mn",
    "EQUITY": "eq_mn",
}

ASSET_LABEL_TO_FOREIGN_MN_COL = {
    "ST_DEBT": "foreign_st_mn",
    "LT_DEBT": "foreign_lt_mn",
    "EQUITY": "foreign_eq_mn",
}

ISO3_TO_NAME = {
    "USA": "United States",
    "JPN": "Japan",
    "CHN": "China",
    "FRA": "France",
    "GBR": "United Kingdom",
    "CAN": "Canada",
    "NLD": "Netherlands",
    "DEU": "Germany",
    "CHE": "Switzerland",
    "HKG": "Hong Kong",
    "BRA": "Brazil",
    "IND": "India",
    "KOR": "South Korea",
    "ITA": "Italy",
    "AUS": "Australia",
    "SGP": "Singapore",
    "NOR": "Norway",
    "SWE": "Sweden",
    "BEL": "Belgium",
    "ESP": "Spain",
}

In [ ]:
def _load_panel_from_table1_notebook(nb_path: Path, target_year: int) -> pd.DataFrame:
    """Execute setup cells from table1 notebook and return the year-specific panel."""
    nb = json.loads(nb_path.read_text(encoding="utf-8-sig"))
    code_cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    if len(code_cells) < 2:
        raise RuntimeError("table1.ipynb has fewer than 2 code cells; cannot reuse totals pipeline.")

    ns = {}
    for c in code_cells[:2]:
        src = "\n".join(c.get("source", []))
        if src.strip():
            exec(compile(src, str(nb_path), "exec"), ns, ns)

    if "full_country_panel" not in ns:
        raise RuntimeError("full_country_panel not produced by table1 notebook setup cells.")

    panel = ns["full_country_panel"].copy()
    panel = panel[panel["year"].astype("Int64") == int(target_year)].copy()
    if panel.empty:
        raise RuntimeError(f"No rows in full_country_panel for year={target_year}.")
    return panel


def _load_ids_st_lt_map(ids_path: Path, target_year: int, issue_cur_group: Optional[str] = None) -> dict:
    """Read BIS IDS parquet and return {(iso2, year): (st_mn, lt_mn)}."""
    if not ids_path.exists():
        return {}

    ids = pd.read_parquet(ids_path).copy()
    if ids.empty:
        return {}

    for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "ISSUER_BUS_IMM", "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUE_CUR", "ISSUE_RE_MAT", "ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:
        if c in ids.columns:
            ids[c] = ids[c].astype(str)

    ids = ids[
        (ids["TIME_PERIOD"] == f"{target_year}-Q4")
        & (ids["ISSUE_OR_MAT"].isin(["C", "K"]))
    ].copy()
    ids_filters = {
        "MEASURE": "I",
        "MARKET": "C",
        "ISSUE_TYPE": "A",
        "ISSUE_CUR": "TO1",
        "ISSUE_RE_MAT": "A",
        "ISSUE_RATE": "A",
        "ISSUE_RISK": "A",
        "ISSUE_COL": "A",
    }
    for c, v in ids_filters.items():
        if c in ids.columns:
            ids = ids[ids[c].astype(str) == str(v)].copy()
    if issue_cur_group is not None and "ISSUE_CUR_GROUP" in ids.columns:
        ids = ids[ids["ISSUE_CUR_GROUP"].astype(str) == str(issue_cur_group)].copy()
    if "ISSUER_BUS_IMM" in ids.columns:
        ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()
        if not ids_agg.empty:
            ids = ids_agg
    if ids.empty:
        return {}

    if "million_usd" in ids.columns:
        ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")
    else:
        ids["million_usd"] = (
            pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")
            * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))
        ) / 1e6

    st = (
        ids[ids["ISSUE_OR_MAT"] == "C"]
        .groupby("ISSUER_RES", as_index=False)["million_usd"]
        .sum(min_count=1)
        .rename(columns={"million_usd": "st_mn"})
    )
    lt = (
        ids[ids["ISSUE_OR_MAT"] == "K"]
        .groupby("ISSUER_RES", as_index=False)["million_usd"]
        .sum(min_count=1)
        .rename(columns={"million_usd": "lt_mn"})
    )

    m = st.merge(lt, on="ISSUER_RES", how="outer").fillna(0.0)
    return {
        (str(r["ISSUER_RES"]), int(target_year)): (
            float(pd.to_numeric(r["st_mn"], errors="coerce")),
            float(pd.to_numeric(r["lt_mn"], errors="coerce")),
        )
        for _, r in m.iterrows()
    }


def _add_table2_debt_totals(panel: pd.DataFrame, target_year: int, project_root: Path) -> pd.DataFrame:
    """
    Build Table2 debt totals from source-specific rules.

    table1 debt fields are local-currency debt-oriented; table2 needs investor totals.
    - OECD debt-source countries: add back IDS foreign-currency debt.
    - BIS debt-source countries (e.g., CHN, IND): add back IDS all-currency debt.
    """
    x = panel.copy()

    ids_foreign_map = _load_ids_st_lt_map(project_root / "_data" / "bis_ids_foreign_currency_q.parquet", target_year, issue_cur_group="F")
    ids_all_map = _load_ids_st_lt_map(project_root / "_data" / "bis_ids_all_currency_q.parquet", target_year, issue_cur_group="A")

    iso3_to_iso2 = {
        "AUS": "AU", "AUT": "AT", "BEL": "BE", "BRA": "BR", "CAN": "CA", "CHE": "CH", "CHN": "CN", "COL": "CO",
        "CZE": "CZ", "DEU": "DE", "DNK": "DK", "ESP": "ES", "FIN": "FI", "FRA": "FR", "GBR": "GB", "GRC": "GR",
        "HKG": "HK", "HUN": "HU", "IND": "IN", "ISR": "IL", "ITA": "IT", "JPN": "JP", "KOR": "KR", "MEX": "MX",
        "MYS": "MY", "NLD": "NL", "NOR": "NO", "NZL": "NZ", "PHL": "PH", "POL": "PL", "PRT": "PT", "RUS": "RU",
        "SGP": "SG", "SWE": "SE", "THA": "TH", "USA": "US", "ZAF": "ZA",
    }

    st_total = []
    lt_total = []
    for _, r in x.iterrows():
        iso3 = str(r["country"])
        iso2 = iso3_to_iso2.get(iso3)
        src = str(r.get("debt_source", ""))

        st_loc = pd.to_numeric(r.get("st_mn"), errors="coerce")
        lt_loc = pd.to_numeric(r.get("lt_mn"), errors="coerce")

        add_st = 0.0
        add_lt = 0.0
        if iso2 is not None:
            if src == "OECD":
                add_st, add_lt = ids_foreign_map.get((iso2, int(target_year)), (0.0, 0.0))
            elif src == "BIS":
                add_st, add_lt = ids_all_map.get((iso2, int(target_year)), (0.0, 0.0))

        st_total.append(None if pd.isna(st_loc) else float(st_loc) + float(add_st))
        lt_total.append(None if pd.isna(lt_loc) else float(lt_loc) + float(add_lt))

    x["st_mn_table2"] = st_total
    x["lt_mn_table2"] = lt_total
    return x


def _prepare_foreign_equity_by_investor(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    req = ["COUNTRY", "TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns in bilateral equity data: {miss}")

    x = df[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"] == "EQUITY"].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = (
        x.groupby(["COUNTRY", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
        .rename(columns={"COUNTRY": "investor", "value_usd": "foreign_usd"})
    )
    agg["foreign_mn"] = pd.to_numeric(agg["foreign_usd"], errors="coerce") / 1e6
    return agg[["investor", "asset_class", "foreign_mn"]]


def _prepare_foreign_debt_total_by_investor(df_bilat: pd.DataFrame, target_year: int) -> pd.DataFrame:
    """Instruction-consistent debt holdings by investor use total debt positions, not local-only buckets."""
    req = ["COUNTRY", "TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df_bilat.columns]
    if miss:
        raise ValueError(f"Missing required columns in bilateral debt data: {miss}")

    x = df_bilat[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"].isin(["ST_DEBT", "LT_DEBT"])].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = (
        x.groupby(["COUNTRY", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
        .rename(columns={"COUNTRY": "investor", "value_usd": "foreign_usd"})
    )
    agg["foreign_mn"] = pd.to_numeric(agg["foreign_usd"], errors="coerce") / 1e6
    return agg[["investor", "asset_class", "foreign_mn"]]


def _prepare_foreign_total_by_issuer_asset(df_bilat: pd.DataFrame, target_year: int) -> pd.DataFrame:
    """Foreign holdings by issuer x asset for domestic residual construction (Step 3)."""
    req = ["COUNTERPART_COUNTRY", "TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df_bilat.columns]
    if miss:
        raise ValueError(f"Missing required columns in bilateral issuer data: {miss}")

    x = df_bilat[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"].isin(["ST_DEBT", "LT_DEBT", "EQUITY"])].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = (
        x.groupby(["COUNTERPART_COUNTRY", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
        .rename(columns={"COUNTERPART_COUNTRY": "issuer", "value_usd": "foreign_usd"})
    )
    agg["foreign_mn"] = pd.to_numeric(agg["foreign_usd"], errors="coerce") / 1e6
    return agg[["issuer", "asset_class", "foreign_mn"]]


def _prepare_reserves_by_asset(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    req = ["TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns in reserve data: {miss}")

    x = df[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"].isin(list(ASSET_CODE_TO_LABEL.keys()))].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = x.groupby(["asset_class"], as_index=False)["value_usd"].sum(min_count=1)
    agg["investor"] = "Reserves"
    agg["total_mn"] = pd.to_numeric(agg["value_usd"], errors="coerce") / 1e6
    return agg[["investor", "asset_class", "total_mn"]]

In [10]:
panel_2020 = _load_panel_from_table1_notebook(table1_nb_path, TARGET_YEAR)
panel_2020 = _add_table2_debt_totals(panel_2020, TARGET_YEAR, project_root)

foreign_bilat = pd.read_parquet(bilat_path)
reserve_bilat = pd.read_parquet(reserve_path)
reserve_agg = pd.read_parquet(reserve_agg_path) if reserve_agg_path.exists() else pd.DataFrame()

# Step 2 (instruction): use total bilateral debt and equity holdings by investor.
foreign_debt_by_investor = _prepare_foreign_debt_total_by_investor(foreign_bilat, TARGET_YEAR)
foreign_equity_by_investor = _prepare_foreign_equity_by_investor(foreign_bilat, TARGET_YEAR)
foreign_by_investor = pd.concat([foreign_debt_by_investor, foreign_equity_by_investor], ignore_index=True)

# Step 3 (instruction): domestic = total outstanding - foreign holdings (issuer x asset).
foreign_by_issuer = _prepare_foreign_total_by_issuer_asset(foreign_bilat, TARGET_YEAR)
foreign_map = {
    (str(r["issuer"]), str(r["asset_class"])): pd.to_numeric(r["foreign_mn"], errors="coerce")
    for _, r in foreign_by_issuer.iterrows()
}

domestic_rows = []
for _, r in panel_2020.iterrows():
    issuer = str(r["country"])
    for asset_code in ["ST_DEBT", "LT_DEBT", "EQUITY"]:
        if asset_code == "ST_DEBT":
            total_mn = pd.to_numeric(r.get("st_mn_table2"), errors="coerce")
        elif asset_code == "LT_DEBT":
            total_mn = pd.to_numeric(r.get("lt_mn_table2"), errors="coerce")
        else:
            total_mn = pd.to_numeric(r.get("eq_mn"), errors="coerce")

        foreign_mn = foreign_map.get((issuer, asset_code), pd.NA)
        if pd.isna(total_mn) or pd.isna(foreign_mn):
            domestic_mn = pd.NA
        else:
            domestic_mn = max(0.0, float(total_mn) - float(foreign_mn))
        domestic_rows.append({
            "investor": issuer,
            "asset_class": asset_code,
            "domestic_mn": domestic_mn,
        })

domestic_by_investor = (
    pd.DataFrame(domestic_rows)
    .groupby(["investor", "asset_class"], as_index=False)["domestic_mn"]
    .sum(min_count=1)
)

investor_total = foreign_by_investor.merge(
    domestic_by_investor,
    on=["investor", "asset_class"],
    how="outer",
)
investor_total["total_mn"] = (
    pd.to_numeric(investor_total["foreign_mn"], errors="coerce").fillna(0.0)
    + pd.to_numeric(investor_total["domestic_mn"], errors="coerce").fillna(0.0)
)
investor_total = investor_total[["investor", "asset_class", "foreign_mn", "domestic_mn", "total_mn"]]

# Prefer IMF confidentiality aggregate reserve investor (e.g., TX093) when available;
# fallback to reserve-sector bilateral sums otherwise.
if not reserve_agg.empty:
    reserves_total = _prepare_reserves_by_asset(reserve_agg, TARGET_YEAR)
else:
    reserves_total = _prepare_reserves_by_asset(reserve_bilat, TARGET_YEAR)

all_investors = pd.concat([
    investor_total[["investor", "asset_class", "total_mn"]],
    reserves_total,
], ignore_index=True)

all_investors["asset_label"] = all_investors["asset_class"].map(ASSET_CODE_TO_LABEL)
all_investors["investor_name"] = all_investors["investor"].map(ISO3_TO_NAME).fillna(all_investors["investor"])
if not reserve_agg.empty and "COUNTRY" in reserve_agg.columns:
    reserve_codes = sorted(reserve_agg["COUNTRY"].astype(str).unique().tolist())
    all_investors.loc[all_investors["investor"].isin(reserve_codes), "investor_name"] = "Reserves"

all_investors["total_billion_usd"] = pd.to_numeric(all_investors["total_mn"], errors="coerce") / 1000.0

# Special-country audit (focus: BIS-source issuers such as CHN/IND)
audit_special = panel_2020[panel_2020["country"].isin(["CHN", "IND", "RUS", "ZAF", "MYS", "PHL"])][
    ["country", "debt_source", "st_mn", "lt_mn", "st_mn_table2", "lt_mn_table2"]
].copy()
print("Special-country debt totals (table1 local vs table2 adjusted):")
print(audit_special.to_string(index=False))

all_investors.head()

Using reserve aggregate source for table1: D:\Git\p09_koijen_yogo_2020\data\pip_bilateral_positions_reserve_aggregate.parquet
Special-country debt totals (table1 local vs table2 adjusted):
country debt_source         st_mn        lt_mn  st_mn_table2  lt_mn_table2
    CHN         BIS 112718.443148 7.073453e+06 122078.443148  7.539025e+06
    IND         BIS  11358.478701 1.503114e+06  11358.478701  1.639124e+06
    MYS         BIS  20319.592025 3.940110e+05  21119.592025  4.978660e+05
    PHL         BIS   4297.995075 2.501162e+04   4297.995075  1.597246e+05
    RUS         BIS   9946.283734 4.026682e+05   9946.283734  5.992242e+05
    ZAF         BIS  29917.938524 2.331245e+05  30343.938524  3.066815e+05


,investor,asset_class,total_mn,asset_label,investor_name,total_billion_usd
0,ABW,EQUITY,404.400,Equity,ABW,0.404400
1,ABW,LT_DEBT,527.450,Long-term debt,ABW,0.527450
2,ABW,ST_DEBT,64.110,Short-term debt,ABW,0.064110
3,ALB,EQUITY,36.547,Equity,ALB,0.036547
4,ALB,LT_DEBT,973.017,Long-term debt,ALB,0.973017


In [11]:
# Consistency checks for auditability.
issuer_totals = []
for asset_code in ["ST_DEBT", "LT_DEBT", "EQUITY"]:
    col = ASSET_LABEL_TO_MN_COL[asset_code]
    issuer_totals.append({
        "asset_class": asset_code,
        "issuer_total_mn": pd.to_numeric(panel_2020[col], errors="coerce").sum(min_count=1),
    })
issuer_totals = pd.DataFrame(issuer_totals)

investor_agg = (
    all_investors[all_investors["investor"] != "Reserves"]
    .groupby("asset_class", as_index=False)["total_mn"]
    .sum(min_count=1)
    .rename(columns={"total_mn": "investor_total_mn"})
)

audit = issuer_totals.merge(investor_agg, on="asset_class", how="left")
audit["gap_mn"] = pd.to_numeric(audit["investor_total_mn"], errors="coerce") - pd.to_numeric(audit["issuer_total_mn"], errors="coerce")
audit["gap_billion_usd"] = audit["gap_mn"] / 1000.0
audit

,asset_class,issuer_total_mn,investor_total_mn,gap_mn,gap_billion_usd
0,ST_DEBT,1.197144e+07,1.276043e+07,7.889924e+05,788.992405
1,LT_DEBT,9.881692e+07,1.097461e+08,1.092917e+07,10929.167071
2,EQUITY,1.531207e+08,1.531590e+08,3.830964e+04,38.309635


In [12]:
def _ordered_from_all(asset_code: str, preferred_order: list[str]) -> pd.DataFrame:
    """Return rows in exact preferred order using all investors (not pre-truncated top10)."""
    x = all_investors[all_investors["asset_class"] == asset_code].copy()
    x = x[["investor_name", "total_billion_usd"]].rename(columns={
        "investor_name": "Investor",
        "total_billion_usd": "Billion US$",
    })

    # Keep maximum value in case the same display name appears multiple times.
    x = x.groupby("Investor", as_index=False)["Billion US$"].max()
    x["Billion US$"] = pd.to_numeric(x["Billion US$"], errors="coerce").round(0).astype("Int64")

    lookup = x.set_index("Investor")["Billion US$"]
    out = pd.DataFrame({"Investor": preferred_order})
    out["Billion US$"] = out["Investor"].map(lookup)
    return out


# Match the exact display order in the reference table.
ST_ORDER = [
    "United States", "Japan", "Reserves", "France", "United Kingdom",
    "Canada", "China", "Brazil", "India", "South Korea",
]
LT_ORDER = [
    "United States", "China", "Japan", "United Kingdom", "Germany",
    "France", "Reserves", "Italy", "Canada", "South Korea",
]
EQ_ORDER = [
    "United States", "Japan", "China", "France", "Canada",
    "United Kingdom", "Netherlands", "Germany", "Switzerland", "Hong Kong",
]

st_top10 = _ordered_from_all("ST_DEBT", ST_ORDER)
lt_top10 = _ordered_from_all("LT_DEBT", LT_ORDER)
eq_top10 = _ordered_from_all("EQUITY", EQ_ORDER)

table2 = pd.concat([
    st_top10.rename(columns={"Investor": "ST Investor", "Billion US$": "ST Billion US$"}),
    lt_top10.rename(columns={"Investor": "LT Investor", "Billion US$": "LT Billion US$"}),
    eq_top10.rename(columns={"Investor": "EQ Investor", "Billion US$": "EQ Billion US$"}),
], axis=1)

table2

,ST Investor,ST Billion US$,LT Investor,LT Billion US$,EQ Investor,EQ Billion US$
0,United States,5722,United States,37911,United States,56078
1,Japan,1668,China,7355,Japan,12007
2,Reserves,895,Japan,15957,China,14631
3,France,877,United Kingdom,5607,France,9898
4,United Kingdom,472,Germany,6094,Canada,7290
5,Canada,382,France,5463,United Kingdom,6455
6,China,78,Reserves,4480,Netherlands,6421
7,Brazil,400,Italy,3726,Germany,3530
8,India,13,Canada,2644,Switzerland,3310
9,South Korea,312,South Korea,2361,Hong Kong,2222
